# Population to Hexagons
### This notebook contains a minimal example of using accessX to prepare hex grids and estimate population per hex for Amsterdam, Athens, and Milan.

In [ ]:
import accessx as acx
import osmnx as ox
import matplotlib.pyplot as plt
from pathlib import Path

### Load Area of Interest

In [ ]:
epsg_ams = 28992
epsg_ath = 2100
epsg_mil = 32632

data_path = Path("../data/case_studies/population")
area_path = data_path / "area"
raster_path = data_path / "raster"
results_path = data_path / "results"

for folder in [data_path, area_path, raster_path, results_path]:
    folder.mkdir(parents=True, exist_ok=True)

gdf_ams = ox.geocode_to_gdf("Amsterdam, Netherlands")
gdf_ath = ox.geocode_to_gdf("Municipality of Athens, Greece")
gdf_mil = ox.geocode_to_gdf("Milan, Italy")

# Uncomment to read AOIs from disk instead of geocoding.
# gdf_ams = acx.read_gdf(area_path / "aoi_amsterdam.geojson")
# gdf_ath = acx.read_gdf(area_path / "aoi_athens.geojson")
# gdf_mil = acx.read_gdf(area_path / "aoi_milan.geojson")

cities = [
    {"name": "amsterdam", "label": "Amsterdam", "aoi": gdf_ams, "epsg": epsg_ams},
    {"name": "athens", "label": "Athens", "aoi": gdf_ath, "epsg": epsg_ath},
    {"name": "milan", "label": "Milan", "aoi": gdf_mil, "epsg": epsg_mil},
]

for city in cities:
    acx.save_gdf(city["aoi"], area_path / f"aoi_{city['name']}.geojson")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

gdfs = [gdf_ams, gdf_ath, gdf_mil]
titles = ["Amsterdam", "Athens", "Milan"]

for ax, gdf, title in zip(axes, gdfs, titles):
    gdf.plot(ax=ax, facecolor="none", edgecolor="black")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()


### Make Hex Grid

In [ ]:
hexes = {}

for city in cities:
    hexes[city["name"]] = acx.make_hex_grid(aoi=city["aoi"], resolution=9)
    acx.save_gdf(hexes[city["name"]], area_path / f"hexes_{city['name']}.geojson")

# Uncomment to read hex grids from disk instead of recomputing.
# hexes = {
#     city["name"]: acx.read_gdf(area_path / f"hexes_{city['name']}.geojson")
#     for city in cities
# }


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

gdfs = [hexes["amsterdam"], hexes["athens"], hexes["milan"]]
titles = ["Amsterdam", "Athens", "Milan"]

for ax, gdf, title in zip(axes, gdfs, titles):
    gdf.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.4)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()


### Download WorldPop Raster

In [ ]:
worldpop_rasters = {}

for city in cities:
    worldpop_rasters[city["name"]] = acx.get_worldpop_raster(
        aoi=city["aoi"],
        year=2020,
        clip=True,
        save_path=raster_path / f"worldpop_{city['name']}_2020_clipped.tif",
    )

# Uncomment to read the raster paths directly if they already exist on disk.
# worldpop_rasters = {
#     city["name"]: raster_path / f"worldpop_{city['name']}_2020_clipped.tif"
#     for city in cities
# }

worldpop_rasters


### Map Population to Hexagons

In [ ]:
hexes_with_population = {}

for city in cities:
    hexes_with_population[city["name"]] = acx.map_population_to_hexes(
        hexes[city["name"]],
        worldpop_rasters[city["name"]],
        metric_crs=city["epsg"],
    )
    acx.save_gdf(
        hexes_with_population[city["name"]],
        results_path / f"hexes_population_{city['name']}.geojson",
    )

# Uncomment to read hex-level population from disk instead of recomputing.
# hexes_with_population = {
#     city["name"]: acx.read_gdf(results_path / f"hexes_population_{city['name']}.geojson")
#     for city in cities
# }


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

gdfs = [hexes_with_population["amsterdam"], hexes_with_population["athens"], hexes_with_population["milan"]]
titles = ["Amsterdam", "Athens", "Milan"]

for ax, gdf, title in zip(axes, gdfs, titles):
    gdf.plot(column="population", ax=ax, legend=True)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
hexes_with_population["amsterdam"][["hex_id", "population"]].head(), hexes_with_population["athens"][["hex_id", "population"]].head(), hexes_with_population["milan"][["hex_id", "population"]].head()
